## 🎯 Learning Objectives
* Design and implement a multi-agent system using AutoGen's `ConversableAgent` and `UserProxyAgent`.
* Configure agents with specific roles and capabilities, including code generation and execution.
* Orchestrate agent communication within a `GroupChat` to collaboratively solve a complex problem.
* Develop strategies for agents to handle and recover from code execution errors.
* Evaluate the effectiveness of an agent team in solving a practical programming task.


### ADV02-L04: Exercise: Build a Code-Writing and Execution Team

#### Task Description

In this exercise, you will leverage your knowledge of AutoGen to construct a collaborative agent team capable of writing, executing, and debugging Python code to solve a given problem. Your team will consist of at least three agents: a `Coder`, an `Executor`, and a `UserProxyAgent` acting as the orchestrator and problem setter.

**The Problem:**

Your agent team needs to analyze a list of sales transactions. Each transaction is a dictionary containing an `'item'` (string) and a `'price'` (float). The team's goal is to write Python code that:
1.  Calculates the **total revenue** from all transactions.
2.  Identifies the **item with the highest total revenue**.
3.  Outputs these two results clearly.

#### Requirements

1.  **Agent Roles:**
    *   **`Coder` Agent:** Responsible for generating Python code to solve the problem. Its system message should guide it to produce clean, runnable code.
    *   **`Executor` Agent:** Responsible for executing the Python code provided by the `Coder`. It must be configured to run code and report the output, including any errors, back to the team.
    *   **`UserProxyAgent` (Orchestrator):** This agent will initiate the conversation with the problem statement and manage the overall flow. It should be configured to allow for automated interaction (`human_input_mode="NEVER"`) for testing, but you can set it to `"ALWAYS"` during development for interactive debugging.

2.  **Collaboration:**
    *   The agents must interact within an AutoGen `GroupChat` to facilitate communication and task delegation.
    *   The `UserProxyAgent` should initiate the chat by presenting the problem to the `Coder`.
    *   The `Coder` should propose code, and the `Executor` should run it.
    *   If the `Executor` encounters an error, it must report it, and the `Coder` should attempt to debug and correct the code based on the feedback.

3.  **Code Execution:**
    *   Ensure the `Executor` agent is properly configured with `code_execution_config` to execute Python code in a sandboxed environment (e.g., using Docker or a local environment).

4.  **Output:**
    *   The final output from the agent team should clearly state the total revenue and the item with the highest total revenue.

#### Evaluation Criteria

*   **Correctness:** Does the agent team correctly calculate the total revenue and identify the highest-revenue item?
*   **Robustness:** Can the team handle and recover from initial coding errors?
*   **Efficiency:** Is the solution achieved in a reasonable number of turns?
*   **Clarity:** Are the agent roles well-defined and their interactions logical?
*   **AutoGen Usage:** Proper and effective use of AutoGen features (`ConversableAgent`, `UserProxyAgent`, `GroupChat`, `code_execution_config`).


In [ ]:
# Setup Code

import os
import autogen

# --- Configuration for LLM --- 
# Ensure you have your API key set as an environment variable.
# For OpenAI, it's `OPENAI_API_KEY`.
# For other providers, adjust accordingly.

# In 2026, we assume robust local LLMs or highly optimized cloud APIs are common.
# For this exercise, we'll use a placeholder for a powerful LLM.
llm_config = {
    "config_list": [
        {
            "model": "gpt-4o-2024-05-13", # Or a similar powerful model available in 2026
            "api_key": os.environ.get("OPENAI_API_KEY")
        }
        # You might add configurations for local models like Ollama or other cloud providers here
        # {"model": "ollama/llama3", "base_url": "http://localhost:11434/v1", "api_key": "ollama"}
    ],
    "temperature": 0.1, # Keep temperature low for deterministic code generation
    "timeout": 120
}

# --- Problem Data --- 
# This is the data your agents will analyze.
# In a real-world scenario, this might be loaded from a file or database.

sales_transactions = [
    {"item": "Laptop", "price": 1200.00},
    {"item": "Mouse", "price": 25.00},
    {"item": "Keyboard", "price": 75.00},
    {"item": "Laptop", "price": 1500.00},
    {"item": "Monitor", "price": 300.00},
    {"item": "Mouse", "price": 30.00},
    {"item": "Webcam", "price": 50.00},
    {"item": "Monitor", "price": 350.00},
    {"item": "Laptop", "price": 1300.00},
    {"item": "Keyboard", "price": 80.00}
]

# Convert the data to a string format that can be easily embedded into the prompt
# or passed as a variable to the agents.
# For simplicity, we'll pass it as a string in the initial prompt.
# In more advanced scenarios, agents might have access to data files or databases.
problem_data_str = str(sales_transactions)

print("Setup complete. Sales transactions data prepared.")
print(f"Sample data: {sales_transactions[0]}")


#### Your Implementation

Now it's your turn to build the agent team. Based on the requirements above, define your `Coder`, `Executor`, and `UserProxyAgent`. Configure their system messages, capabilities, and orchestrate their interaction using `GroupChat`.

Your goal is to have the team successfully analyze the `sales_transactions` data and output the total revenue and the highest-revenue item.

```python
# Your code goes here

# Example structure:
# coder = autogen.ConversableAgent(...)
# executor = autogen.ConversableAgent(...)
# user_proxy = autogen.UserProxyAgent(...)

# groupchat = autogen.GroupChat(...)
# manager = autogen.GroupChatManager(...)

# user_proxy.initiate_chat(manager, message="...")
```


In [ ]:
# --- Reference Solution --- 

# 1. Define the Coder Agent
# The Coder's primary role is to generate Python code based on the problem description.
# It should be instructed to provide runnable code and be ready to debug.
coder = autogen.ConversableAgent(
    name="Coder",
    llm_config=llm_config,
    system_message=(
        "You are an expert Python programmer. Your task is to write clean, efficient, and correct Python code "
        "to solve data analysis problems. When asked to analyze data, you will be provided with the data structure. "
        "You should output only the code, enclosed in a code block. "
        "If the code fails, you will receive error messages and should debug and correct your code. "
        "Your final output should be the requested results, not just the code."
    ),
    human_input_mode="NEVER", # Coder doesn't need human input directly
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").upper(),
    code_execution_config=False # Coder does not execute code itself
)

# 2. Define the Executor Agent
# The Executor's role is to run the code provided by the Coder and report the results.
# It must have code execution enabled.
executor = autogen.ConversableAgent(
    name="Executor",
    llm_config=llm_config,
    system_message=(
        "You are a Python code executor. Your role is to run the Python code provided by the Coder. "
        "You will report the output of the code execution, including any errors. "
        "If the code runs successfully, provide the output. If there's an error, provide the full traceback. "
        "Do not try to fix the code, just execute it and report."
    ),
    human_input_mode="NEVER", # Executor doesn't need human input directly
    code_execution_config={
        "work_dir": "coding", # Directory to save and run code files
        "use_docker": True,   # Use Docker for sandboxed execution (recommended for security and consistency)
        "timeout": 60         # Timeout for code execution in seconds
    }
)

# 3. Define the User Proxy Agent (Orchestrator)
# The User Proxy initiates the conversation and can provide human feedback if needed.
# For this exercise, we set human_input_mode to NEVER for automated testing.
user_proxy = autogen.UserProxyAgent(
    name="Admin",
    human_input_mode="NEVER", # Set to "ALWAYS" for interactive debugging
    max_consecutive_auto_reply=10, # Allow up to 10 auto-replies before human intervention (if enabled)
    is_termination_msg=lambda x: "TERMINATE" in x.get("content", "").upper(),
    code_execution_config=False # User Proxy does not execute code itself
)

# 4. Create a GroupChat
# The GroupChat manages the communication flow between the agents.
# The manager will decide which agent speaks next based on the conversation history.
groupchat = autogen.GroupChat(
    agents=[user_proxy, coder, executor],
    messages=[],
    max_round=15, # Limit the number of turns to prevent infinite loops
    speaker_selection_method="auto" # AutoGen decides who speaks next
)

# 5. Create a GroupChatManager
# The manager orchestrates the group chat.
manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config
)

# 6. Initiate the Chat
# The User Proxy starts the conversation by posing the problem to the team.
# We embed the problem data directly into the prompt for simplicity.

problem_statement = (
    f"Analyze the following list of sales transactions: {problem_data_str}. "
    "Each transaction is a dictionary with 'item' and 'price'. "
    "Your task is to write Python code to: "
    "1. Calculate the total revenue from all transactions. "
    "2. Identify the item with the highest total revenue. "
    "3. Output these two results clearly. "
    "The final answer should include the total revenue and the highest revenue item. "
    "Once you have provided the final answer, please say 'TERMINATE'."
)

print("\n--- Initiating Agent Conversation ---\n")

chat_result = user_proxy.initiate_chat(
    manager,
    message=problem_statement
)

print("\n--- Agent Conversation Finished ---\n")

# You can inspect the chat_result for details about the conversation
# print(chat_result.summary)
# print(chat_result.chat_history)
